In [ ]:
训练阶段，推理阶段(加载模型) 应分成两个脚本

my_project/
├── model_definition.py  # 专门存放 IncomeMLP 类的代码
├── train.py             # 训练脚本：fit scaler, 训练模型, 保存 pth 和 joblib
├── predict.py           # 推理脚本：加载 pth 和 joblib, 接收输入, 给出预测
├── scaler.joblib        # 存好的缩放器
└── model_weights.pth    # 存好的模型权重

In [2]:
import torch
import torch.nn as nn
import numpy as np
import joblib

# 1. 设定设备 (一定要和当前环境匹配)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. “造身体”：定义一个和训练时一模一样的模型结构
# 必须完全一致，否则权重填不进去
model = nn.Sequential(
    nn.Linear(5, 32), # 这里的 5 是 len(numeric_cols)
    nn.ReLU(),
    nn.Linear(32, 1),
    nn.Sigmoid()
).to(device)

# 3. “导记忆”：加载保存好的权重
weights_path = "model_weights.pth"
model.load_state_dict(torch.load(weights_path, map_location=device))
scaler = joblib.load('scaler.joblib') # 这样 scaler 就有定义了，且拥有训练集的记忆

# 4. “准备考试”：切换到评估模式
model.eval()

# ==========================================
# 5. 实现预测（运行）
# ==========================================

# 模拟一个新用户的数据 (年龄39, 教育13, 资本增加2174, 资本减少0, 工时40)
new_data = np.array([[39, 13, 2174, 0, 40]])

new_data_scaled = scaler.transform(new_data) # 必须标准化！
input_tensor = torch.FloatTensor(new_data_scaled).to(device)

with torch.no_grad():
    prediction_prob = model(input_tensor)
    prediction_label = (prediction_prob > 0.5).float()

print(f"\n预测概率: {prediction_prob.item():.4f}")
print(f"预测结果: {'高收入' if prediction_label.item() == 1 else '低收入'}")


预测概率: 0.3954
预测结果: 低收入
